# CODE FRAUDE

In [ ]:
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from collections import defaultdict

## CLASSIFICA VAR IMPORTANTI

In [ ]:
# # =========================================================
# # DATA
# # =========================================================
# df = pd.read_csv("/kaggle/input/datasets/monfo00/training-5/training_4.csv")
# cli_info = pd.read_csv("/kaggle/input/datasets/monfo00/training-5/cli_var_sel.csv")

# # Join su ID_CLIENTE
# df = df.merge(cli_info, on="ID_CLIENTE", how="left")

# TARGET = "isFraud"
# y = df[TARGET]
# X = df.drop(columns=[TARGET, "ID_CLIENTE"])  # tolgo anche l'id, non è una feature

# print(X.shape)  # controlla il numero di colonne dopo il join

# non_numeric_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
# print(f"Trovate {len(non_numeric_cols)} colonne non numeriche:")
# print(non_numeric_cols)

# X=X.drop(columns=non_numeric_cols)


In [ ]:
# mem_before = X.memory_usage(deep=True).sum() / 1024**2
# print(f"Memoria prima: {mem_before:.2f} MB")

# # float64 -> float32
# float_cols = X.select_dtypes(include=["float64"]).columns
# X[float_cols] = X[float_cols].astype("float32")

# # int64 -> int32
# int_cols = X.select_dtypes(include=["int64"]).columns
# X[int_cols] = X[int_cols].astype("int32")

# # Memoria dopo la conversione
# mem_after = X.memory_usage(deep=True).sum() / 1024**2
# print(f"Memoria dopo: {mem_after:.2f} MB")
# print(f"Risparmio: {mem_before - mem_after:.2f} MB ({100*(mem_before-mem_after)/mem_before:.1f}%)")

# # Controllo dei tipi
# print(X.dtypes.value_counts())

In [ ]:
# def recursive_feature_importance(
#     X, y,
#     n_iterations=40,
#     sample_size=250,
#     n_splits=5,
#     random_state=19,
#     xgb_params=None
# ):
#     all_cols = X.columns.tolist()
#     n_cols = len(all_cols)
#     sample_size = min(sample_size, n_cols)

#     if xgb_params is None:
#         xgb_params = {
#             "objective": "binary:logistic",
#             "eval_metric": "auc",
#             "max_depth": 6,
#             "learning_rate": 0.05,
#             "subsample": 0.8,
#             "colsample_bytree": 0.8,
#             "tree_method": "hist",
#             "random_state": random_state,
#         }

#     importance_sum = defaultdict(float)
#     appearance_count = defaultdict(int)

#     rng = np.random.RandomState(random_state)

#     for it in range(n_iterations):
#         sampled_cols = rng.choice(all_cols, size=sample_size, replace=False)
#         X_sub = X[sampled_cols]

#         skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
#         fold_importances = defaultdict(list)
#         aucs = []

#         for fold, (tr_idx, val_idx) in enumerate(skf.split(X_sub, y)):
#             X_tr, X_val = X_sub.iloc[tr_idx], X_sub.iloc[val_idx]
#             y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

#             dtrain = xgb.DMatrix(X_tr, label=y_tr)
#             dval = xgb.DMatrix(X_val, label=y_val)

#             model = xgb.train(
#                 xgb_params,
#                 dtrain,
#                 num_boost_round=500,
#                 evals=[(dval, "val")],
#                 early_stopping_rounds=100,   # <-- FUORI dal dict, come argomento
#                 verbose_eval=False,
#             )

#             preds = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
#             auc = roc_auc_score(y_val, preds)
#             aucs.append(auc)

#             gain_dict = model.get_score(importance_type="gain")
#             for col, gain in gain_dict.items():
#                 fold_importances[col].append(gain)

#         mean_auc = np.mean(aucs)

#         for col, gains in fold_importances.items():
#             importance_sum[col] += np.mean(gains)
#             appearance_count[col] += 1

#         print(f"[Iter {it+1}/{n_iterations}] sample_size={sample_size} | AUC medio={mean_auc:.5f}")

#     results = pd.DataFrame({
#         "feature": list(importance_sum.keys()),
#         "importance_sum": [importance_sum[c] for c in importance_sum],
#         "n_appearances": [appearance_count[c] for c in importance_sum],
#     })
#     results["importance_mean"] = results["importance_sum"] / results["n_appearances"]

#     results = results.sort_values(
#         by=["n_appearances", "importance_mean"], ascending=[False, False]
#     ).reset_index(drop=True)

#     return results



# results = recursive_feature_importance(
#     X, y,
#     n_iterations=30,
#     sample_size=500,
#     n_splits=5,
#     random_state=19
# )

# top_700 = results.head(700)["feature"].tolist()
# print(f"Trovate {len(top_700)} feature top su {X.shape[1]} totali")

# results.to_csv("feature_importance_recursive.csv", index=False)


# #X=X[top_300].copy()

## BAYESIAN OPT

In [ ]:
# =========================================================
# DATA
# =========================================================
df = pd.read_csv("/kaggle/input/datasets/monfo00/training-5/train_5.csv")

TARGET = "isFraud"
y = df[TARGET]
X = df.drop(columns=[TARGET, "ID_CLIENTE"])  # tolgo anche l'id, non è una feature

print(X.shape)  # controlla il numero di colonne dopo il join

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=19
)

# =========================================================
# OBJECTIVE
# =========================================================

def objective(trial):

    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",

        # GPU
        "tree_method": "hist",
        "device": "cuda",

        # Hyperparameters
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.001,
            0.3,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        "min_child_weight": trial.suggest_float(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            0.95
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.2,
            0.9
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0,
            5
        ),

        "max_bin": trial.suggest_int(
            "max_bin",
            256,
            2048
        )
    }

    scores = []
    best_iters = []

    # =====================================================
    # CROSS VALIDATION
    # =====================================================

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):

        X_tr = X.iloc[tr_idx]

        y_tr = y.iloc[tr_idx]

        X_va = X.iloc[va_idx]
        
        y_va = y.iloc[va_idx]

        # =================================================
        # GPU DATASET
        # =================================================

        dtrain = xgb.DMatrix(
            X_tr,
            label=y_tr
        )
        
        dvalid = xgb.DMatrix(
            X_va,
            label=y_va
        )

        # =================================================
        # TRAIN
        # =================================================

        bst = xgb.train(
            params,
            dtrain,
            num_boost_round=1000,
            evals=[(dvalid, "val")],
            early_stopping_rounds=100,
            verbose_eval=False
        )

        # =================================================
        # PREDICT
        # =================================================

        preds = bst.predict(dvalid)

        auc = roc_auc_score(
            y_va,
            preds
        )

        scores.append(auc)

        best_iters.append(
            bst.best_iteration
        )

    # =====================================================
    # SCORE
    # =====================================================

    mean_auc = np.mean(scores)

    std_auc = np.std(scores)

    trial.set_user_attr(
        "std_auc",
        float(std_auc)
    )

    trial.set_user_attr(
        "mean_iter",
        int(np.mean(best_iters))
    )

    return mean_auc - 0.5 * std_auc


# =========================================================
# STUDY
# =========================================================

study = optuna.create_study(
    direction="maximize",

    sampler=optuna.samplers.TPESampler(
        n_startup_trials=30,
        multivariate=True,
        seed=19
    ),

    pruner=optuna.pruners.MedianPruner(
        n_warmup_steps=20
    )
)

# =========================================================
# OPTIMIZATION
# =========================================================

study.optimize(
    objective,
    n_trials=300,
    n_jobs=1
)

# =========================================================
# RESULTS
# =========================================================

best = study.best_trial

print("\n==============================")
print("BEST RESULT")
print("==============================")

print(f"Score     : {study.best_value:.6f}")
print(f"STD       : {best.user_attrs['std_auc']:.6f}")
print(f"Best iter : {best.user_attrs['mean_iter']}")

print("\nBest params:")

for k, v in best.params.items():
    print(f"{k}: {v}")

# =========================================================
# GPU CHECK
# =========================================================

print("\nXGBoost version:", xgb.__version__)
print(xgb.build_info())

## TEST CON TOP PARAM

In [ ]:
# =========================================================
# TRAIN FINAL MODEL WITH BEST OPTUNA PARAMS
# =========================================================

best_params = study.best_trial.params.copy()

best_params.update({
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "device": "cuda"
})

best_iter = study.best_trial.user_attrs["mean_iter"]

print("Best iteration:", best_iter)
print("Training final model...")

dtrain_full = xgb.DMatrix(
    X,
    label=y
)

final_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=best_iter
)

print("Final model trained.")

# =========================================================
# LOAD TEST
# =========================================================

X_test = pd.read_csv(
    "/kaggle/input/datasets/monfo00/training-5/test_5.csv"
).drop(columns="ID_CLIENTE")

test_ids = pd.read_csv("/kaggle/input/datasets/monfo00/training-5/test_transaction.csv")["TransactionID"].copy()


print("Test shape:", X_test.shape)

# =========================================================
# PREDICT PROBABILITY OF FRAUD
# =========================================================

dtest = xgb.DMatrix(X_test)

preds = final_model.predict(dtest)

print("\nPrediction summary:")
print(pd.Series(preds).describe())

# =========================================================
# EXPORT SUBMISSION
# =========================================================

submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nSubmission saved as submission.csv")
print(submission.head())